# Métodos Estatísticos aplicados aos setores censitários de Foz do Iguaçu

Versão pública e reproduzível do trabalho da disciplina **Métodos Estatísticos Avançados**.

**Pergunta:** Como se distribuem a população e a densidade populacional entre os setores censitários de Foz do Iguaçu?

## 1. Preparação dos dados

Baixe `BR_setores_CD2022.csv` e `PR_setores_CD2022.gpkg` conforme `data/README.md` e ajuste os caminhos abaixo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from scipy.stats import norm, t, chi2, probplot

CSV_PATH = '../data/BR_setores_CD2022.csv'
GPKG_PATH = '../data/PR_setores_CD2022.gpkg'

df = pd.read_csv(CSV_PATH)
foz = df[df['CD_MUN'] == 4108304].copy()

foz = foz.rename(columns={
    'v0001': 'POPULACAO',
    'v0002': 'DOMICILIOS',
    'v0003': 'DOM_PARTICULARES',
    'v0004': 'DOM_COLETIVOS',
    'v0005': 'MEDIA_MORADORES',
    'v0006': 'PERC_DOM_IMPUTADOS',
    'v0007': 'DOM_PART_OCUPADOS'
})

setores_pr = gpd.read_file(GPKG_PATH)
setores_foz = setores_pr[setores_pr['CD_MUN'].astype(str) == '4108304'].copy()

foz['CD_SETOR'] = foz['CD_SETOR'].astype(str)
setores_foz['CD_SETOR'] = setores_foz['CD_SETOR'].astype(str)
foz = foz.merge(setores_foz[['CD_SETOR','geometry']], on='CD_SETOR', how='left', validate='one_to_one')
foz = gpd.GeoDataFrame(foz, geometry='geometry', crs=setores_foz.crs)
foz['DENSIDADE_POP'] = foz['POPULACAO'] / foz['AREA_KM2']

print('Setores:', len(foz))
print('População total:', int(foz['POPULACAO'].sum()))


## 2. Qualidade dos dados

In [ ]:
checagens = pd.Series({
    'CD_SETOR duplicado': foz['CD_SETOR'].duplicated().sum(),
    'AREA_KM2 <= 0': (foz['AREA_KM2'] <= 0).sum(),
    'POPULACAO < 0': (foz['POPULACAO'] < 0).sum(),
    'PERC_DOM_IMPUTADOS fora de 0-100': ((foz['PERC_DOM_IMPUTADOS'] < 0) | (foz['PERC_DOM_IMPUTADOS'] > 100)).sum(),
    'Geometria ausente': foz.geometry.isna().sum(),
    'Geometria inválida': (~foz.geometry.is_valid).sum()
})
print(checagens)

print('\nSITUACAO ausente:')
display(foz.loc[foz['SITUACAO'].isna(), ['CD_SETOR','CD_SIT','AREA_KM2','POPULACAO']])

zero_pop = foz[foz['POPULACAO'] == 0].copy()
print('\nSetores com população zero:', len(zero_pop))
print('Área total desses setores (km²):', zero_pop['AREA_KM2'].sum())


## 3. Estatística descritiva

In [ ]:
def separatriz_curso(serie, proporcao):
    valores = np.sort(serie.dropna().to_numpy())
    n = len(valores)
    pos = proporcao * (n + 1)
    if pos <= 1: return valores[0]
    if pos >= n: return valores[-1]
    if float(pos).is_integer(): return valores[int(pos)-1]
    frac = pos - np.floor(pos)
    if np.isclose(frac, 0.5):
        i = int(np.floor(pos))
        return (valores[i-1] + valores[i]) / 2
    return valores[int(np.floor(pos + 0.5))-1]

for var in ['POPULACAO','DENSIDADE_POP']:
    s = foz[var]
    q1, med, q3 = [separatriz_curso(s,p) for p in (0.25,0.5,0.75)]
    p10, p90 = [separatriz_curso(s,p) for p in (0.10,0.90)]
    media = s.mean(); dp = s.std(ddof=0); cv = dp/media*100
    print(f'\n{var}')
    print('média=', round(media,2), 'mediana=', round(med,2), 'Q1=', round(q1,2), 'Q3=', round(q3,2))
    print('DP=', round(dp,2), 'CV=', round(cv,2), '%')
    print('P10=', round(p10,2), 'P90=', round(p90,2))


### 3.1 Distribuições de frequência

In [ ]:
bins_pop = np.arange(0, 2200 + 220, 220)
bins_dens = np.arange(0, 40000 + 4000, 4000)

fig, ax = plt.subplots(figsize=(8,5))
ax.hist(foz['POPULACAO'], bins=bins_pop, edgecolor='black')
ax.set(title='População por setor censitário', xlabel='Habitantes', ylabel='Frequência')
plt.show()

fig, ax = plt.subplots(figsize=(8,5))
ax.hist(foz['DENSIDADE_POP'], bins=bins_dens, edgecolor='black')
ax.set(title='Densidade populacional por setor', xlabel='hab./km²', ylabel='Frequência')
plt.show()


### 3.2 Assimetria, curtose e boxplots

In [ ]:
def moda_agrupada_midpoint(serie, bins):
    cats = pd.cut(serie, bins=bins, right=False, include_lowest=True)
    classe = cats.value_counts().idxmax()
    return (classe.left + classe.right) / 2

def resumo_forma(serie, bins):
    media = serie.mean(); dp = serie.std(ddof=0)
    mo = moda_agrupada_midpoint(serie,bins)
    q1 = separatriz_curso(serie,0.25); q3 = separatriz_curso(serie,0.75)
    p10 = separatriz_curso(serie,0.10); p90 = separatriz_curso(serie,0.90)
    assimetria = (media - mo)/dp
    curtose = (q3-q1)/(2*(p90-p10))
    return assimetria, curtose

print('POPULACAO:', resumo_forma(foz['POPULACAO'], bins_pop))
print('DENSIDADE_POP:', resumo_forma(foz['DENSIDADE_POP'], bins_dens))

fig, ax = plt.subplots(figsize=(7,4))
ax.boxplot(foz['POPULACAO'])
ax.set(title='Boxplot — população por setor', ylabel='Habitantes')
plt.show()

fig, ax = plt.subplots(figsize=(7,4))
ax.boxplot(foz['DENSIDADE_POP'])
ax.set(title='Boxplot — densidade populacional', ylabel='hab./km²')
plt.show()


### 3.3 Mapas coropléticos

In [ ]:
foz.explore(column='POPULACAO', cmap='YlOrRd', tiles='OpenStreetMap', legend=True)

limites = [0,4000,8000,12000,16000,float('inf')]
labels = ['0 a <4000','4000 a <8000','8000 a <12000','12000 a <16000','16000 ou mais']
foz['CLASSE_DENS_MAPA'] = pd.cut(foz['DENSIDADE_POP'], bins=limites, labels=labels, right=False, include_lowest=True)
foz.explore(column='CLASSE_DENS_MAPA', categorical=True, cmap='YlOrRd', tiles='OpenStreetMap', legend=True)


## 4. Análise amostral

In [ ]:
N = len(foz)
Z = 1.96
resultados_n = []
for var in ['POPULACAO','DENSIDADE_POP']:
    media = foz[var].mean(); sigma = foz[var].std(ddof=0)
    for erro_pct in [0.05,0.10,0.15]:
        E = erro_pct * media
        n_calc = N*Z**2*sigma**2 / (E**2*(N-1) + Z**2*sigma**2)
        resultados_n.append([var, erro_pct*100, n_calc, np.ceil(n_calc)])

dimensionamento = pd.DataFrame(resultados_n, columns=['Variável','Erro relativo (%)','n calculado','n adotado'])
display(dimensionamento.round(2))

n = 187
amostra_aas = foz.sample(n=n, replace=False, random_state=42).copy()


### 4.1 Amostragem aleatória estratificada proporcional

In [ ]:
foz['ESTRATO_AMOSTRAL'] = foz['SITUACAO'].fillna('Massas de água (reservatório de Itaipu)')
tamanho = foz['ESTRATO_AMOSTRAL'].value_counts().sort_index()
aloc_exata = tamanho/N*n
aloc = np.floor(aloc_exata).astype(int)
restos = (aloc_exata-aloc).sort_values(ascending=False)
for estrato in restos.index[:n-aloc.sum()]: aloc.loc[estrato]+=1

tabela_alocacao = pd.DataFrame({'N_i':tamanho,'n_i teórico':aloc_exata,'n_i adotado':aloc})
display(tabela_alocacao.round(2))

partes=[]
for j,(estrato,n_i) in enumerate(aloc.items()):
    grupo=foz[foz['ESTRATO_AMOSTRAL']==estrato]
    partes.append(grupo.sample(n=int(n_i), replace=False, random_state=42+j))
amostra_aae=pd.concat(partes).copy()
pesos=tamanho/N

def media_estratificada(amostra,var):
    return (amostra.groupby('ESTRATO_AMOSTRAL')[var].mean()*pesos).sum()


### 4.2 Comparação de AAS e AAE por simulação

In [ ]:
parametros={v:foz[v].mean() for v in ['POPULACAO','DENSIDADE_POP']}
resultados=[]
for rep in range(1000):
    aas=foz.sample(n=n, replace=False, random_state=1000+rep)
    partes=[]
    for j,(estrato,n_i) in enumerate(aloc.items()):
        grupo=foz[foz['ESTRATO_AMOSTRAL']==estrato]
        partes.append(grupo.sample(n=int(n_i), replace=False, random_state=100000+rep*10+j))
    aae=pd.concat(partes)
    for var in parametros:
        est_aas=aas[var].mean(); est_aae=media_estratificada(aae,var); mu=parametros[var]
        resultados += [
            [rep+1,'AAS',var,est_aas,abs(est_aas-mu)/mu*100],
            [rep+1,'AAE',var,est_aae,abs(est_aae-mu)/mu*100]
        ]

simulacao=pd.DataFrame(resultados, columns=['Repetição','Método','Variável','Estimativa','Erro relativo (%)'])
display(simulacao.groupby(['Variável','Método'])['Erro relativo (%)'].agg(['mean','median','std']).round(2))


## 5. Estimação

In [ ]:
N_est=len(foz); n_est=len(amostra_aas); alpha=0.05
zcrit=norm.ppf(1-alpha/2); tcrit=t.ppf(1-alpha/2, n_est-1)
fpc=np.sqrt((N_est-n_est)/(N_est-1))

linhas=[]
for var in ['POPULACAO','DENSIDADE_POP']:
    xbar=amostra_aas[var].mean(); mu=foz[var].mean(); sigma=foz[var].std(ddof=0); s=amostra_aas[var].std(ddof=1)
    for metodo,crit,desvio in [('Z - σ conhecido',zcrit,sigma),('t - σ desconhecido',tcrit,s)]:
        for ajuste,fator in [('Sem correção finita',1.0),('Com correção finita',fpc)]:
            margem=crit*desvio/np.sqrt(n_est)*fator
            linhas.append([var,metodo,ajuste,xbar,xbar-margem,xbar+margem,mu,(xbar-margem)<=mu<=(xbar+margem)])

ic_media=pd.DataFrame(linhas,columns=['Variável','Método','Ajuste','Estimativa','LI 95%','LS 95%','Média populacional','Contém parâmetro?'])
display(ic_media.round(2))


### 5.1 Normalidade e intervalo qui-quadrado

In [ ]:
probplot(amostra_aas['POPULACAO'], dist='norm', plot=plt); plt.title('Q-Q plot — população'); plt.show()
probplot(amostra_aas['DENSIDADE_POP'], dist='norm', plot=plt); plt.title('Q-Q plot — densidade'); plt.show()

s2=amostra_aas['POPULACAO'].var(ddof=1); gl=n_est-1
qinf=chi2.ppf(alpha/2,gl); qsup=chi2.ppf(1-alpha/2,gl)
li_var=gl*s2/qsup; ls_var=gl*s2/qinf
print('IC 95% variância:', li_var, ls_var)
print('IC 95% desvio padrão:', np.sqrt(li_var), np.sqrt(ls_var))


## 6. Síntese

- A população por setor é aproximadamente simétrica, mas apresenta elevada dispersão.
- A densidade é mais heterogênea e apresenta assimetria positiva.
- A AAE proporcional apresentou pequena vantagem média sobre a AAS nas simulações.
- A correção para população finita foi relevante nos intervalos de confiança devido à elevada fração amostral.

Os valores consolidados e a interpretação detalhada estão no README do repositório.